# 📕 03-神经网络与完整训练

> 本笔记是 PyTorch 学习路线的第三部分，涵盖神经网络核心组件、激活函数数学原理、模型保存与完整的端到端训练流程。
>
> **学习路径**: ① [PyTorch基础](./01-PyTorch基础.ipynb) → ② [图像处理与数据加载](./02-图像处理与数据加载.ipynb) → ③ 本笔记（神经网络与训练）

## 📋 目录
- [1. 激活函数与数学原理](#1-激活函数与数学原理)
- [2. 神经网络核心组件](#2-神经网络核心组件)
- [3. 损失函数与优化器](#3-损失函数与优化器)
- [4. 模型保存与加载](#4-模型保存与加载)
- [5. 完整训练流程](#5-完整训练流程)
- [6. 总结与速查](#6-总结与速查)

**运行环境**：`pip install torch torchvision numpy matplotlib tensorboard`

---

## 1. 激活函数与数学原理

> 💡 **为什么需要激活函数？**
>
> 线性模型的组合仍然是线性的。激活函数引入**非线性**，使神经网络能够学习任意复杂的映射关系。

### 1.1 ReLU（修正线性单元）— 最常用 ⭐

$$\text{ReLU}(x) = \max(0, x) = \begin{cases} 0, & x \leq 0 \\ x, & x > 0 \end{cases}$$

- **值域**：$[0, +\infty)$
- **导数**：$\text{ReLU}'(x) = \begin{cases} 0, & x \leq 0 \\ 1, & x > 0 \end{cases}$
- **优点**：计算简单，正区梯度恒为 1，缓解梯度消失
- **缺点**：负区梯度为 0（"死亡 ReLU"问题）
- **适用**：CNN 隐藏层（首选）

In [ ]:
# ==================== ReLU 可视化 ====================
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 500)
y = np.maximum(0, x)

plt.figure(figsize=(7, 4))
plt.plot(x, y, linewidth=2, color='green')
plt.title('ReLU: f(x) = max(0, x)', fontsize=13, fontweight='bold')
plt.xlabel('x'); plt.ylabel('ReLU(x)')
plt.grid(True, alpha=0.3, linestyle='--')
plt.ylim(-1, 5); plt.axhline(0, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

### 1.2 Sigmoid — 经典激活函数

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

- **值域**：$(0, 1)$，可解释为概率
- **导数**：$\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$
- **缺点**：两端梯度趋近于 0（梯度消失）
- **适用**：二分类输出层

In [ ]:
# ==================== Sigmoid 可视化 ====================
x = np.linspace(-5, 5, 500)
y = 1 / (1 + np.exp(-x))

plt.figure(figsize=(7, 4))
plt.plot(x, y, linewidth=2, color='blue')
plt.title('Sigmoid: σ(x) = 1 / (1 + e⁻ˣ)', fontsize=13, fontweight='bold')
plt.xlabel('x'); plt.ylabel('σ(x)')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3, linestyle='--')
plt.ylim(-0.1, 1.1); plt.tight_layout(); plt.show()

### 1.3 Tanh — 双曲正切

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

- **值域**：$(-1, 1)$，零中心化
- **导数**：$\tanh'(x) = 1 - \tanh^2(x)$
- **优点**：零中心化，比 Sigmoid 训练更快
- **适用**：RNN 隐藏层

In [ ]:
# ==================== 三大激活函数对比 ====================
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
x = np.linspace(-8, 8, 500)

# ReLU
axes[0].plot(x, np.maximum(0, x), 'green', linewidth=2)
axes[0].set_title('ReLU'); axes[0].grid(alpha=0.3)
axes[0].set_ylim(-1, 6)

# Sigmoid
sig = 1 / (1 + np.exp(-x))
axes[1].plot(x, sig, 'blue', linewidth=2)
axes[1].set_title('Sigmoid'); axes[1].grid(alpha=0.3)
axes[1].set_ylim(-0.1, 1.1)

# Tanh
axes[2].plot(x, np.tanh(x), 'red', linewidth=2)
axes[2].set_title('Tanh'); axes[2].grid(alpha=0.3)
axes[2].set_ylim(-1.2, 1.2)

for ax in axes: ax.set_xlabel('x')
plt.tight_layout(pad=2); plt.show()

**激活函数选择指南**：

| 场景 | 推荐 | 原因 |
|------|------|------|
| CNN 隐藏层 | **ReLU** | 计算简单，梯度不消失 |
| RNN 隐藏层 | **Tanh** | 零中心化，梯度流稳定 |
| 二分类输出 | **Sigmoid** | 输出概率 $[0,1]$ |
| 多分类输出 | **Softmax** | 概率分布，各类和为 1 |
| 需要更平滑 | **SiLU/GELU** | 某些场景优于 ReLU |

---

## 2. 神经网络核心组件

### 2.1 nn.Module — 自定义网络基类

所有自定义模型都必须继承 `nn.Module`，它提供参数管理、模型保存/加载等核心功能。

```python
# model(x) 的实际执行流程:
# model(x) → __call__(x) → 执行 hooks → forward(x) → 执行 hooks → 返回结果
```

> **为什么用 `model(x)` 而不是 `model.forward(x)`？**
>
> `__call__` 在调用 `forward()` 前后自动执行 hooks、参数管理、设备切换等操作。

In [ ]:
# ==================== 最简自定义模型 ====================
import torch
from torch import nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()  # 必须调用！使模型获得参数管理能力
    
    def forward(self, x):
        return x + 1  # 前向传播计算逻辑

model = SimpleModel()
x = torch.tensor(5.0)
print(f"输入: {x}, 输出: {model(x)}")
# 调用 model(x) 实际执行的是 __call__，内部再调用 forward()

### 2.2 卷积层 (Conv2d)

卷积层通过可学习的卷积核在图像上滑动来提取局部特征。

**输出尺寸公式**：
$$\text{output} = \left\lfloor\frac{\text{input} + 2 \times \text{padding} - \text{kernel}}{\text{stride}} + 1\right\rfloor$$

In [ ]:
# ==================== 卷积运算示例 ====================
import torch
import torch.nn.functional as F

# 5x5 输入矩阵
input_tensor = torch.tensor([[1,2,3,4,2],
                              [3,4,5,6,7],
                              [5,6,7,8,9],
                              [7,8,9,10,11],
                              [3,4,5,6,7]], dtype=torch.float32)

# 3x3 卷积核
kernel = torch.tensor([[1,2,1],[0,1,0],[2,0,1]], dtype=torch.float32)

# 调整为 4D: (batch, channels, height, width)
input_4d = input_tensor.unsqueeze(0).unsqueeze(0)  # [1, 1, 5, 5]
kernel_4d = kernel.unsqueeze(0).unsqueeze(0)       # [1, 1, 3, 3]

# 不同参数的卷积运算
print(f"输入形状: {input_4d.shape}")
print(f"卷积核形状: {kernel_4d.shape}")
print()

out1 = F.conv2d(input_4d, kernel_4d, stride=1)
print(f"stride=1, padding=0 → 输出: {out1.shape}, 公式: (5-3)/1+1=3")

out2 = F.conv2d(input_4d, kernel_4d, stride=2)
print(f"stride=2, padding=0 → 输出: {out2.shape}, 公式: (5-3)/2+1=2")

out3 = F.conv2d(input_4d, kernel_4d, stride=1, padding=1)
print(f"stride=1, padding=1 → 输出: {out3.shape}, 公式: (5+2-3)/1+1=5")

### 2.3 池化层 (MaxPool2d)

池化层用于降维、减少参数量，同时增强特征的鲁棒性。

In [ ]:
# ==================== 最大池化示例 ====================
from torch import nn

test_input = torch.tensor([[1,2,3,4,2],
                            [3,4,5,6,7],
                            [5,6,7,8,9],
                            [7,8,9,10,11],
                            [3,4,5,6,7]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

maxpool = nn.MaxPool2d(kernel_size=3, ceil_mode=True)
output = maxpool(test_input)

print(f"输入形状: {test_input.shape} (5x5)")
print(f"池化输出: {output.shape} (3x3, ceil_mode向上取整)")
print(f"\n池化作用: 降维 → 减少计算量 → 增强特征不变性")

### 2.4 核心组件速查表

| 组件 | 类 | 作用 | 关键参数 |
|------|---|------|----------|
| **卷积** | `nn.Conv2d` | 特征提取 | `in_channels, out_channels, kernel_size, stride, padding` |
| **池化** | `nn.MaxPool2d` | 降维增强特征 | `kernel_size, stride` |
| **ReLU** | `nn.ReLU` | 引入非线性 | — |
| **Sigmoid** | `nn.Sigmoid` | 压缩到 [0,1] | — |
| **全连接** | `nn.Linear` | 分类输出 | `in_features, out_features` |
| **展平** | `nn.Flatten` | 多维→一维 | — |
| **Dropout** | `nn.Dropout(p)` | 随机失活，防过拟合 | `p` (丢弃概率) |

---

## 3. 损失函数与优化器

### 3.1 损失函数

损失函数衡量预测值与真实值之间的差距，是优化器的优化目标。

In [ ]:
# ==================== 常用损失函数 ====================
from torch import nn

# ── L1Loss (平均绝对误差) ──
pred = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[1.0, 2.0, 5.0]])
loss_l1 = nn.L1Loss()(pred, target)
print(f"L1Loss:  {loss_l1.item():.4f}  (|3-5|/3 = 2/3)")

# ── MSELoss (均方误差) ──
loss_mse = nn.MSELoss()(pred, target)
print(f"MSELoss: {loss_mse.item():.4f}  ((3-5)²/3 = 4/3)")

# ── CrossEntropyLoss (交叉熵) ⭐ 分类首选 ──
# 输入: 未归一化的 logits; 标签: 类别索引
logits = torch.tensor([[0.1, 0.2, 0.7]])  # 3 个类别的 logits
label = torch.tensor([2])                   # 真实类别索引 = 2
loss_ce = nn.CrossEntropyLoss()(logits, label)
print(f"CrossEntropyLoss: {loss_ce.item():.4f}  ← 多分类任务首选")

print("\n损失函数选择指南:")
print("  分类任务 → CrossEntropyLoss")
print("  回归任务 → MSELoss 或 L1Loss")

### 3.2 优化器 (SGD)

优化器根据梯度更新模型参数。SGD（随机梯度下降）是最基础的优化算法。

**训练三步曲**：
1. `optimizer.zero_grad()` — 清零历史梯度
2. `loss.backward()` — 反向传播计算梯度
3. `optimizer.step()` — 根据梯度更新参数

In [ ]:
# ==================== SGD 优化器演示 ====================
# 这里仅演示核心流程，不依赖外部数据集
import torch
from torch import nn

# 定义简单模型
model = nn.Sequential(
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 3)
)

# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 模拟一个训练批次
imgs = torch.randn(8, 10)   # 8个样本, 每个10维
targets = torch.randint(0, 3, (8,))  # 8个标签, 3个类别

# 1. 前向传播
output = model(imgs)
loss = loss_fn(output, targets)
print(f"前向传播 - Loss: {loss.item():.4f}")

# 2. 反向传播三步骤
optimizer.zero_grad()   # ① 清零梯度
loss.backward()         # ② 计算梯度
optimizer.step()        # ③ 更新参数

print("\n训练三步曲:")
print("  ① optimizer.zero_grad()  → 清零历史梯度")
print("  ② loss.backward()       → 反向传播计算梯度")
print("  ③ optimizer.step()      → 根据梯度更新参数")

---

## 4. 模型保存与加载

### 4.1 两种方式对比

| 对比项 | 方式一：完整模型 | 方式二：仅参数 ⭐推荐 |
|--------|----------------|--------------------|
| 保存内容 | 结构 + 参数 | 仅参数 (state_dict) |
| 文件大小 | 较大 | 较小 |
| 安全性 | 执行反序列化代码 | 仅加载数据，更安全 |
| 适用场景 | 自己的模型 | 官方预训练模型 |

In [ ]:
# ==================== 方式一：保存完整模型 ====================
import torch
import torchvision

model = torchvision.models.vgg16(pretrained=False)

# 保存
torch.save(model, "model_full.pth")

# 加载
loaded_model = torch.load("model_full.pth", weights_only=False)

print("✅ 方式一: 保存完整模型 (结构+参数)")
print("   torch.save(model, 'file.pth')")
print("   torch.load('file.pth', weights_only=False)")

In [ ]:
# ==================== 方式二：仅保存参数（官方推荐）====================
import torch
import torchvision

# 保存参数
model = torchvision.models.vgg16(pretrained=False)
torch.save(model.state_dict(), "model_params.pth")

# 加载参数
model = torchvision.models.vgg16(pretrained=False)  # 先创建结构
model.load_state_dict(torch.load("model_params.pth", weights_only=True))  # 再加载参数

print("✅ 方式二: 仅保存参数 (推荐)")
print("   torch.save(model.state_dict(), 'file.pth')")
print("   model.load_state_dict(torch.load('file.pth', weights_only=True))")

---

## 5. 完整训练流程

将以上所有组件整合，实现 CIFAR10 分类的端到端训练。

In [1]:
# ==================== CIFAR10 完整训练流程 ====================
# 运行说明:
#   1. 首次运行下载 CIFAR10 (~170MB)
#   2. 在 TensorBoard 中查看: tensorboard --logdir logs/cifar10

import torch
import torchvision
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader

# ─── 步骤 1: 定义设备 ───
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 使用设备: {device}")

# ─── 步骤 2: 加载数据集 ───
print("📥 加载 CIFAR10 数据集...")
train_data = torchvision.datasets.CIFAR10(
    root="./dataSet", train=True, download=True,
    transform=torchvision.transforms.ToTensor()
)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
print(f"   训练集: {len(train_data)} 张, {len(train_loader)} 个批次")

# ─── 步骤 3: 定义模型 ───
class CIFAR10Net(nn.Module):
    """CNN 用于 CIFAR10 分类 (32x32 RGB → 10类)"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 32x32x3 → 16x16x32
            nn.Conv2d(3, 32, 5, 1, 2), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2: 16x16x32 → 8x8x32
            nn.Conv2d(32, 32, 5, 1, 2), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3: 8x8x32 → 4x4x64
            nn.Conv2d(32, 64, 5, 1, 2), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),          # 64*4*4 = 1024
            nn.Linear(1024, 64),
            nn.ReLU(),
            nn.Linear(64, 10)      # 10 个类别
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CIFAR10Net().to(device)
print(f"\n🧠 模型结构:")
print(model)

# ─── 步骤 4: 定义损失函数和优化器 ───
loss_fn = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# ─── 步骤 5: TensorBoard ───
writer = SummaryWriter("logs/cifar10")

# ─── 步骤 6: 训练循环 ───
num_epochs = 5
print(f"\n🚀 开始训练 {num_epochs} 个 epoch...")

for epoch in range(num_epochs):
    running_loss = 0.0
    
    for batch_idx, (imgs, targets) in enumerate(train_loader):
        imgs, targets = imgs.to(device), targets.to(device)
        
        # 前向传播
        outputs = model(imgs)
        loss = loss_fn(outputs, targets)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # 记录平均损失
    avg_loss = running_loss / len(train_loader)
    writer.add_scalar("train/loss", avg_loss, epoch)
    print(f"   Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

writer.close()
print(f"\n✅ 训练完成! TensorBoard: tensorboard --logdir logs/cifar10")

🔧 使用设备: cpu
📥 加载 CIFAR10 数据集...
   训练集: 50000 张, 782 个批次

🧠 模型结构:
CIFAR10Net(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=10, bias=True)
  )
)

🚀 开始训练 5 个 epoch...
   Epoch 1/5 | Loss: 2.2967
   Epoch 2/5 | Loss: 2.1413


KeyboardInterrupt: 

### 5.1 训练流程模板（可直接复用）

```python
# 1. 数据准备
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 2. 模型
model = MyModel().to(device)

# 3. 损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 4. 训练循环
for epoch in range(num_epochs):
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        
        outputs = model(imgs)           # 前向传播
        loss = loss_fn(outputs, targets) # 计算损失
        
        optimizer.zero_grad()           # 清零梯度
        loss.backward()                 # 反向传播
        optimizer.step()                # 更新参数
```

---

## 6. 总结与速查

### Tensor 形状变换

- `view()` / `reshape()`: 改变张量形状，总元素数不变
- `-1`: 自动推断该维度大小

### 完整学习路线回顾

| 阶段 | 笔记 | 核心内容 |
|------|------|----------|
| 基础 | ① [PyTorch基础](./01-PyTorch基础.ipynb) | 张量、自动求导、TensorBoard |
| 数据 | ② [图像处理与数据加载](./02-图像处理与数据加载.ipynb) | Transform、Dataset、DataLoader |
| 训练 | ③ 本笔记 | 激活函数、网络组件、损失函数、完整训练流程 |

**🎉 恭喜！你已经学完了 PyTorch 的核心知识，可以开始训练自己的模型了！**